# legal-expand 1.3.1 - Demo interactiva para Google Colab

[![PyPI version](https://img.shields.io/pypi/v/legal-expand.svg?label=PyPI)](https://pypi.org/project/legal-expand/)
[![GitHub](https://img.shields.io/badge/GitHub-686f6c61%2Fpypi--legal--expand-black)](https://github.com/686f6c61/pypi-legal-expand)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/686f6c61/pypi-legal-expand/blob/main/legal_expand_demo.ipynb)

Este notebook muestra la version Python de `legal-expand`: expansion de siglas juridicas espanolas, diagnostico, glosarios, auditoria, CLI, procesamiento de documentos y diccionarios personalizados.

Version objetivo: **legal-expand 1.3.1**.


## 1. Instalacion

En Colab conviene instalar la version exacta publicada para que la demo sea reproducible.


In [ ]:
!pip install -U legal-expand==1.3.1 -q


In [ ]:
import json
from pathlib import Path
from IPython.display import HTML, display

import legal_expand
from legal_expand import (
    ExpansionOptions,
    auditar_texto,
    benchmark_texto,
    buscar_sigla,
    configurar_globalmente,
    expandir_siglas,
    expandir_siglas_detallado,
    extraer_siglas,
    exportar_glosario,
    generar_glosario,
    listar_siglas,
    obtener_configuracion_global,
    obtener_estadisticas,
    obtener_info_diccionario,
    procesar_directorio,
    resetear_configuracion,
)

print('legal-expand version:', legal_expand.__version__)
assert legal_expand.__version__ == '1.3.1'


## 2. Expansion basica

`expandir_siglas()` devuelve texto plano por defecto, anadiendo el significado entre parentesis.


In [ ]:
texto = 'La AEAT revisa el IVA segun el BOE y el art. 123 del CC.'
print(expandir_siglas(texto))


## 3. Variantes dinamicas

La libreria detecta variantes frecuentes sin inflar el diccionario fuente: mayusculas, minusculas y formas con puntos.


In [ ]:
ejemplos = [
    'La AEAT notifica.',
    'La aeat notifica.',
    'La A.E.A.T. notifica.',
    'El art 5 del CC.',
    'El art. 5 del CC.',
]

for ejemplo in ejemplos:
    print('-', expandir_siglas(ejemplo))


## 4. Formatos de salida

`plain` es el formato por defecto. Tambien puedes pedir HTML semantico o salida estructurada.


In [ ]:
muestra = 'La AEAT publica criterios sobre IVA en el BOE.'

print('PLAIN')
print(expandir_siglas(muestra))

print('\nHTML')
html = expandir_siglas(muestra, ExpansionOptions(format='html'))
display(HTML(f'<div style="font-size:16px; line-height:1.6">{html}</div>'))


In [ ]:
estructurado = expandir_siglas(muestra, ExpansionOptions(format='structured'))

print('Texto expandido:', estructurado.expanded_text)
print('Siglas detectadas:', estructurado.stats.total_acronyms_found)
for item in estructurado.acronyms:
    print(f'- {item.acronym}: {item.expansion} [{item.position.start}-{item.position.end}]')


## 5. Diagnostico de omisiones

`expandir_siglas_detallado()` explica por que una sigla se omitio: filtros, repeticion, contexto protegido, ambiguedad o no encontrada.


In [ ]:
texto_diagnostico = (
    'AEAT y BOE aparecen aqui.\n'
    'Contacto: info@aeat.es\n'
    'Codigo: `AEAT`\n'
    'XYZ no esta en el diccionario.\n'
    'AEAT aparece otra vez.'
)

diagnostico = expandir_siglas_detallado(
    texto_diagnostico,
    ExpansionOptions(expand_only_first=True, exclude=['BOE'])
)

print(diagnostico.expanded_text)
print('\nOmisiones:')
for item in diagnostico.omitted_acronyms:
    print(f'- {item.acronym} | {item.reason} | pos={item.position.start}')


## 6. Extraccion sin modificar el texto

`extraer_siglas()` detecta siglas conocidas y candidatos desconocidos. Es util para auditoria documental.


In [ ]:
extraccion = extraer_siglas('AEAT, BOE, XYZ y AEAT de nuevo.')

for item in extraccion.acronyms:
    estado = 'conocida' if item.known else 'desconocida'
    print(
        f'{item.acronym:6} {estado:12} '
        f'ocurrencia {item.occurrence_index}/{item.total_occurrences} '
        f'-> {item.expansion}'
    )


## 7. Glosarios

Desde `1.3.0` ya no hace falta construir glosarios a mano: usa `generar_glosario()` o `exportar_glosario()`.


In [ ]:
texto_glosario = 'La AEAT gestiona IVA e IRPF. El BOE publica el CC.'

print('Entradas de glosario:')
for entry in generar_glosario(texto_glosario):
    print(f'- {entry.acronym}: {entry.expansion} ({entry.count})')

print('\nMarkdown:')
print(exportar_glosario(texto_glosario, 'markdown'))

print('\nCSV:')
print(exportar_glosario(texto_glosario, 'csv'))


## 8. Auditoria completa

`auditar_texto()` combina extraccion, glosario, conocidas/desconocidas, omitidas y repetidas.


In [ ]:
reporte = auditar_texto('AEAT, BOE, XYZ, AEAT y el correo info@boe.es')

print(reporte.to_json(indent=2)[:1200])
print('\nResumen:')
print(reporte.stats)


## 9. CLI en Colab

Al instalar el paquete queda disponible el comando `legal-expand`.


In [ ]:
!legal-expand info


In [ ]:
!printf 'AEAT y BOE\n' | legal-expand --format plain


In [ ]:
%%bash
cat > sentencia.txt <<'TXT'
La AEAT liquida el IVA. El BOE publica la norma. XYZ queda pendiente.
TXT

legal-expand audit sentencia.txt --report-format markdown --output audit.md
legal-expand glossary sentencia.txt --glossary-format csv --output glosario.csv

echo '--- audit.md ---'
cat audit.md
echo '--- glosario.csv ---'
cat glosario.csv


## 10. Documentos y batch

`procesar_directorio()` procesa carpetas de `.txt`, `.md` y `.html`. En HTML expande nodos de texto conservando etiquetas y atributos.


In [ ]:
entrada = Path('demo_docs')
salida = Path('demo_docs_expandidos')
entrada.mkdir(exist_ok=True)

(entrada / 'nota.txt').write_text('La AEAT revisa el IVA publicado en el BOE.\n', encoding='utf-8')
(entrada / 'web.html').write_text(
    '<p>La AEAT revisa el IVA.</p><a href="https://aeat.es">AEAT</a>',
    encoding='utf-8'
)

resultados = procesar_directorio(entrada, salida, ExpansionOptions(), document_format='auto')
for result in resultados:
    print(result.input_path, '->', result.output_path, 'OK=', result.processed)

print('\nTXT expandido:')
print((salida / 'nota.txt').read_text(encoding='utf-8'))

print('HTML expandido:')
print((salida / 'web.html').read_text(encoding='utf-8'))


## 11. Diccionarios personalizados JSON/CSV

Puedes cargar siglas propias sin tocar el diccionario base. No se usan perfiles ni categorias: solo entradas personalizadas directas.


In [ ]:
custom_json = Path('custom_legal_expand.json')
custom_json.write_text(json.dumps([
    {
        'original': 'LXP',
        'significado': 'Legal Expand Python',
        'variants': ['L.X.P.'],
        'source': 'demo-colab',
        'keywords': ['paquete', 'python', 'demo'],
        'priority': 200,
    }
], ensure_ascii=False, indent=2), encoding='utf-8')

opciones_custom = ExpansionOptions(custom_dictionaries=[str(custom_json)])
print(expandir_siglas('LXP y L.X.P. aparecen junto a AEAT.', opciones_custom))

info = obtener_info_diccionario([str(custom_json)])
print(info.to_json(indent=2))


## 12. Busqueda y estadisticas del diccionario


In [ ]:
print('Buscar AEAT:')
print(buscar_sigla('AEAT').to_json(indent=2))

siglas = listar_siglas()
print('Total listadas:', len(siglas))
print('Primeras 20:', siglas[:20])

print('\nEstadisticas:')
print(obtener_estadisticas().to_json(indent=2))


## 13. Configuracion global

La configuracion global permite definir defaults para una aplicacion completa. Reseteala al final de la demo para evitar sorpresas en celdas posteriores.


In [ ]:
from legal_expand import GlobalConfig

configurar_globalmente(GlobalConfig(
    default_options=ExpansionOptions(format='html', expand_only_first=True)
))

print(obtener_configuracion_global().to_json(indent=2))
display(HTML(expandir_siglas('AEAT, AEAT y BOE')))

resetear_configuracion()
print('Configuracion reseteada:', obtener_configuracion_global().to_json())


## 14. Benchmark rapido


In [ ]:
bench = benchmark_texto(
    'La AEAT gestiona el IVA segun el BOE y el CC. ' * 50,
    iterations=100,
)
print(bench.to_json(indent=2))


## 15. Instalacion desde PyPI y enlaces

- PyPI: https://pypi.org/project/legal-expand/
- GitHub: https://github.com/686f6c61/pypi-legal-expand
- npm original: https://www.npmjs.com/package/legal-expand

Para instalar en un proyecto normal:

```bash
pip install legal-expand
```

Para usar la CLI:

```bash
legal-expand info
legal-expand audit documento.txt --report-format markdown
legal-expand batch docs/ docs-expandidos/ --format html
```
